In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
!pip install -q transformers==4.52.4
!pip install -q peft==0.15.2
!pip uninstall -y torchao

# DLGenAI Project Milestone-4


This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.


Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [4]:
import pandas as pd

# Load training data
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Label mapping
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

# Encode labels
train_df["label"] = train_df["answer"].map(label_map)

# Display first few rows
train_df[["answer", "label"]].head()

,answer,label
0,B,1
1,A,0
2,C,2
3,B,1
4,A,0


In [5]:
print("Encoded label at index 150:")
print(train_df.loc[150, "label"])

Encoded label at index 150:
2


Q2. Prompt-Option Formatting


For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [6]:
# Create the formatted input for Option B of row 0
formatted_input = str(train_df.loc[0, "prompt"]) + " [SEP] " + str(train_df.loc[0, "B"])

# Print the formatted text (optional)
print(formatted_input)

# Print its character length
print("Character Length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character Length: 407


**Tokenization for Multiple-Choice Models**

Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length


Since each question has five options, every row becomes five tokenized sequences.


Q3. Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [7]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Row 0
row = train_df.loc[0]

# Create prompt-option pairs
choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for Multiple Choice model
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print(input_ids.shape)
print(attention_mask.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])
torch.Size([1, 5, 128])


Q4. Batch MCQ Tokenization


Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [8]:
batch_size = 16
num_choices = 5
sequence_length = 128

total_token_positions = batch_size * num_choices * sequence_length
print(total_token_positions)

10240


**Multiple-Choice Model Outputs**

AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Q5. Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Get row 0
row = train_df.loc[0]

# Create prompt-option pairs
choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for Multiple Choice model
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

# Forward pass
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Logits shape:", outputs.logits.shape)
print(outputs.logits)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Logits shape: torch.Size([1, 5])
tensor([[0.2165, 0.2404, 0.2580, 0.2390, 0.2318]])


Q6. Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [10]:
print(outputs)

MultipleChoiceModelOutput(loss=None, logits=tensor([[0.2165, 0.2404, 0.2580, 0.2390, 0.2318]]), hidden_states=None, attentions=None)


In [11]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

encoded_label = label_map[train_df.loc[0, "answer"]]

In [12]:
labels = torch.tensor([encoded_label])   # Shape: [1]

In [13]:
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

In [14]:
print(outputs.loss)
print(outputs.loss.shape)
print(outputs.loss.dim())

tensor(1.6063, grad_fn=<NllLossBackward0>)
torch.Size([])
0


**LoRA for Efficient Fine-Tuning**

LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.


Q7. LoRA Trainable Parameters


Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [15]:
from peft import LoraConfig, get_peft_model, TaskType

# Load model
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable Parameters:", trainable_params)

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable Parameters: 295681


**Preparing Data for Hugging Face Trainer**

Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.


Q8. Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [16]:
from datasets import Dataset

# Label encoding
label_map = {"A":0, "B":1, "C":2, "D":3, "E":4}
train_df["label"] = train_df["answer"].map(label_map)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess(example):
    choices = [
        f"{example['prompt']} [SEP] {example['A']}",
        f"{example['prompt']} [SEP] {example['B']}",
        f"{example['prompt']} [SEP] {example['C']}",
        f"{example['prompt']} [SEP] {example['D']}",
        f"{example['prompt']} [SEP] {example['E']}",
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": example["label"],
    }

dataset = Dataset.from_pandas(train_df)
dataset = dataset.map(preprocess)

print(len(dataset[0]["input_ids"]))      # 5
print(len(dataset[0]["input_ids"][0]))   # 128

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

5
128


**Tiny Fine-Tuning and Inference**

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.


Q9. Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [17]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForMultipleChoice

# First 32 rows
train_dataset = Dataset.from_pandas(train_df.head(32))

# Preprocessing
def preprocess(example):
    first_sentences = [example["prompt"]] * 5
    second_sentences = [
        example["A"],
        example["B"],
        example["C"],
        example["D"],
        example["E"],
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        padding="max_length",
        max_length=64,
    )

    tokenized["label"] = label_map[example["answer"]]
    return tokenized

train_dataset = train_dataset.map(preprocess)

train_dataset = train_dataset.remove_columns(
    ["id", "prompt", "A", "B", "C", "D", "E", "answer"]
)

# Data collator
data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Train
trainer.train()

# Answer to Q9
print("Global Step:", trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
1,1.696300
2,1.591700
3,1.507800
4,1.514900


Global Step: 4


Q10. Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [18]:
import torch
import torch.nn.functional as F

# Row 0
row = train_df.loc[0]

# Create prompt-option pairs
choices = [
    [row["prompt"], row["A"]],
    [row["prompt"], row["B"]],
    [row["prompt"], row["C"]],
    [row["prompt"], row["D"]],
    [row["prompt"], row["E"]],
]

# Tokenize
encoding = tokenizer(
    [c[0] for c in choices],
    [c[1] for c in choices],
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

# Add batch dimension
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

# Inference
model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# Softmax probabilities
probs = F.softmax(outputs.logits, dim=1)

print("Probabilities:", probs)
print("Probability of Option E:", probs[0, 4].item())
print("Rounded:", round(probs[0, 4].item(), 4))

Probabilities: tensor([[0.2031, 0.2093, 0.2038, 0.1959, 0.1879]])
Probability of Option E: 0.18786616623401642
Rounded: 0.1879
